# Q1: where do negative and extreme hours sit?

In [ ]:
import duckdb
import matplotlib.pyplot as plt

# read_only so the notebook can never modify the database, and never takes the write lock
# that a `duckdb` CLI session in another terminal would then be refused.
con = duckdb.connect("../data/processed/grid.duckdb", read_only=True)
con.execute("SET TimeZone = 'UTC'")


def q(sql):
    """Run SQL and return a pandas DataFrame, so matplotlib has a table to draw."""
    return con.execute(sql).df()


def heat(grid, xlabel, ylabel, clabel, xstep=1):
    """Draw a pivot table as a heatmap, taking the tick labels from the frame itself.

    imshow otherwise labels the axes 0..n-1, and passing `extent` by hand invites the axes to
    disagree with the data: a grid of 6 years by 12 months drawn with an extent meant for 24
    hours is still a picture, just a wrong one. Reading the labels off grid.index and
    grid.columns makes that impossible.
    """
    fig, ax = plt.subplots(figsize=(9, 4))
    im = ax.imshow(grid, aspect="auto", origin="lower")
    ax.set_xticks(range(0, len(grid.columns), xstep), grid.columns[::xstep])
    ax.set_yticks(range(len(grid.index)), grid.index)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.colorbar(im, ax=ax, label=clabel)
    fig.tight_layout()
    return fig, ax

## G_DOW

In [ ]:
dow = q("""
    SELECT day_of_week,
           100.0 * COUNT(*) FILTER (WHERE price_neg)     / COUNT(*) AS neg_pct,
           100.0 * COUNT(*) FILTER (WHERE extreme_price) / COUNT(*) AS ext_pct
    FROM table_h_price_load GROUP BY 1 ORDER BY 1
""")


fig_d, ax_d = plt.subplots(figsize=(8, 3.5))
ax_d.plot(dow.day_of_week, dow.neg_pct, marker="o", label="negative")
ax_d.plot(dow.day_of_week, dow.ext_pct, marker="o", label="extreme")
ax_d.set_xlabel("day of week")
ax_d.set_ylabel("% of hours in that group")
ax_d.legend()
ax_d.grid(alpha=0.3)
fig_d.tight_layout()

## G_HOUR, G_MONTH and G_YEAR

In [ ]:
hour = q("""
    SELECT hour_date,
           100.0 * COUNT(*) FILTER (WHERE price_neg)     / COUNT(*) AS neg_pct,
           100.0 * COUNT(*) FILTER (WHERE extreme_price) / COUNT(*) AS ext_pct
    FROM table_h_price_load GROUP BY 1 ORDER BY 1
""")

month = q("""
    SELECT month_date,
           100.0 * COUNT(*) FILTER (WHERE price_neg)     / COUNT(*) AS neg_pct,
           100.0 * COUNT(*) FILTER (WHERE extreme_price) / COUNT(*) AS ext_pct
    FROM table_h_price_load GROUP BY 1 ORDER BY 1
""")

year = q("""
    SELECT year_date,
           100.0 * COUNT(*) FILTER (WHERE price_neg)     / COUNT(*) AS neg_pct,
           100.0 * COUNT(*) FILTER (WHERE extreme_price) / COUNT(*) AS ext_pct
    FROM table_h_price_load GROUP BY 1 ORDER BY 1
""")

fig_h, ax_h = plt.subplots(figsize=(8, 3.5))
ax_h.plot(hour.hour_date, hour.neg_pct, marker="o", label="negative")
ax_h.plot(hour.hour_date, hour.ext_pct, marker="o", label="extreme")
ax_h.set_xlabel("hour of day (UTC)")
ax_h.set_ylabel("% of hours in that group")
ax_h.set_xticks(range(0, 24, 2))
ax_h.legend()
ax_h.grid(alpha=0.3)
fig_h.tight_layout()

fig_m, ax_m = plt.subplots(figsize=(8, 3.5))
ax_m.plot(month.month_date, month.neg_pct, marker="o", label="negative")
ax_m.plot(month.month_date, month.ext_pct, marker="o", label="extreme")
ax_m.set_xlabel("month")
ax_m.set_ylabel("% of hours in that group")
ax_m.legend()
ax_m.grid(alpha=0.3)
fig_m.tight_layout()


fig_y, ax_y = plt.subplots(figsize=(8, 3.5))
ax_y.plot(year.year_date, year.neg_pct, marker="o", label="negative")
ax_y.plot(year.year_date, year.ext_pct, marker="o", label="extreme")
ax_y.set_xlabel("year")
ax_y.set_ylabel("% of hours in that group")
ax_y.legend()
ax_y.grid(alpha=0.3)
fig_y.tight_layout()

## G_HOUR_MONTH

In [ ]:
grid_hm_neg = q("""
    SELECT month_date, hour_date,
           100.0 * COUNT(*) FILTER (WHERE price_neg) / COUNT(*) AS neg_pct
    FROM table_h_price_load GROUP BY 1, 2
""").pivot(index="month_date", columns="hour_date", values="neg_pct")

grid_hm_ext = q("""
    SELECT month_date, hour_date,
           100.0 * COUNT(*) FILTER (WHERE extreme_price) / COUNT(*) AS ext_pct
    FROM table_h_price_load GROUP BY 1, 2
""").pivot(index="month_date", columns="hour_date", values="ext_pct")

heat(grid_hm_neg, "hour of day (UTC)", "month", "% of hours negative", xstep=2)
heat(grid_hm_ext, "hour of day (UTC)", "month", "% of hours extreme", xstep=2)

## G_MONTH_YEAR

In [ ]:
grid_my_neg = q("""
    SELECT month_date, year_date,
           100.0 * COUNT(*) FILTER (WHERE price_neg) / COUNT(*) AS neg_pct
    FROM table_h_price_load GROUP BY 1, 2
""").pivot(index="year_date", columns="month_date", values="neg_pct")

grid_my_ext = q("""
    SELECT month_date, year_date,
           100.0 * COUNT(*) FILTER (WHERE extreme_price) / COUNT(*) AS ext_pct
    FROM table_h_price_load GROUP BY 1, 2
""").pivot(index="year_date", columns="month_date", values="ext_pct")

heat(grid_my_neg, "month", "year", "% of hours negative")
heat(grid_my_ext, "month", "year", "% of hours extreme")